# 우체국 금융사기 피해사례 가설 분석
## 0. 데이터 준비

개별 보이스피싱 피해사례를 이용하여 피해자의 특성, 사기유형, 사칭기관과 피해금액의 관계를 확인하고 AI 위험도 판단 후보 변수를 탐색합니다.

이 Notebook은 기존 `가설검정.ipynb`에서 우체국 관련 셀을 분리한 독립 실행용 Notebook입니다.

## 0-1. 환경 준비

Google Drive를 연결하고 데이터 처리에 필요한 `pandas`, 경로 처리를 위한 `Path`만 불러옵니다. 그래프를 만들지 않으므로 폰트 설치 코드는 포함하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 100)

## 0-2. 데이터 경로 설정

프로젝트 루트는 `BASE_DIR` 한 곳에서만 관리합니다. Google Drive에 올린 폴더명이 `이종열`과 다르면 아래 한 줄만 수정하세요.

In [ ]:
BASE_DIR = Path('/content/drive/MyDrive/이종열')
DATA_DIR = BASE_DIR / '데이터'
HYPOTHESIS_DIR = BASE_DIR / '가설'
REFERENCE_DIR = BASE_DIR / '분석기준'

file_paths = {
    '우체국 피해사례': DATA_DIR / 'df_postal.csv',
}

print('프로젝트 루트:', BASE_DIR)
for name, path in file_paths.items():
    print(f'{name}: {path.name} / 존재={path.exists()}')

missing_files = [str(path) for path in file_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('다음 파일을 찾을 수 없습니다:\n' + '\n'.join(missing_files))

## 0-3. CSV 불러오기

`df_postal.csv`는 UTF-8 BOM(`utf-8-sig`)으로 읽습니다. 인코딩을 명시하고, 읽기에 실패하면 오류가 그대로 드러나도록 합니다.

In [ ]:
df_postal_raw = pd.read_csv(file_paths['우체국 피해사례'], encoding='utf-8-sig')

raw_dataframes = {
    '우체국 피해사례': df_postal_raw,
}

for name, df in raw_dataframes.items():
    print(f'{name}: {df.shape}')

### 우체국 데이터의 역할

우체국 데이터는 개별 금융사기 피해사례입니다. 전화 기반 보이스피싱 사례에서 피해자 특성, 사기유형, 사칭기관과 피해금액의 관계를 탐색합니다. 투자사기는 현재 서비스 범위에서 제외합니다.

## 0-4. 원본 데이터 기본 구조 확인

값을 변경하기 전에 `head`, `shape`, 컬럼명, `info`, 기초 통계를 확인합니다. 반복 출력만 줄이기 위해 작은 확인 함수를 사용합니다.

In [ ]:
def show_basic_structure(name, df):
    print('\n' + '=' * 80)
    print(name)
    print('shape:', df.shape)
    print('columns:', df.columns.tolist())
    display(df.head())
    print('\n[info]')
    df.info()
    print('\n[숫자형 기초 통계]')
    display(df.describe())
    print('\n[전체 컬럼 요약]')
    display(df.describe(include='all').transpose())

for name, df in raw_dataframes.items():
    show_basic_structure(name, df)

## 0-5. 결측치 확인

컬럼별 결측 개수와 결측률을 함께 확인합니다. 이 단계에서는 결측 행을 삭제하거나 채우지 않습니다.

In [ ]:
for name, df in raw_dataframes.items():
    missing_report = pd.DataFrame({
        '결측 개수': df.isna().sum(),
        '결측률(%)': (df.isna().mean() * 100).round(2),
    })
    print('\n', name)
    display(missing_report)

## 0-6. 중복 확인

완전히 같은 행과 우체국의 `중복후보` 표시를 구분해 확인합니다. 서로 다른 사건이 같은 값을 가질 수 있으므로 원본에서는 자동 삭제하지 않습니다.

In [ ]:
for name, df in raw_dataframes.items():
    duplicate_count = int(df.duplicated().sum())
    print(f'{name}: 완전중복 추가 행 {duplicate_count}건')
    if duplicate_count > 0:
        display(df[df.duplicated(keep=False)].sort_values(df.columns.tolist()).head(50))

In [ ]:
print('우체국 중복후보 분포')
display(df_postal_raw['중복후보'].value_counts(dropna=False).rename_axis('중복후보').to_frame('건수'))

print('중복후보=True인 행')
display(df_postal_raw[df_postal_raw['중복후보'].eq(True)])

## 0-7. 범주형 값 확인

고유값 수가 작고 가설에 필요한 범주만 확인합니다. 긴 자유서술 컬럼 전체에 `value_counts()`를 적용하지 않습니다.

In [ ]:
postal_category_columns = [
    '연령대', '피해자 성별', '사기유형', '사칭기관',
    '피해구제 신청사유', '접근매체', '전화_보이스피싱', '중복후보',
]

for column in postal_category_columns:
    print(f'\n[{column}] 고유값 수: {df_postal_raw[column].nunique(dropna=False)}')
    display(df_postal_raw[column].value_counts(dropna=False).to_frame('건수'))

## 0-8. dtype 및 값 형식 확인

컬럼명·문자열 앞뒤 공백과 숫자형이어야 할 값의 변환 실패 여부를 확인합니다. 우체국 `피해액` 컬럼은 원 단위입니다.

In [ ]:
for name, df in raw_dataframes.items():
    column_space_count = sum(column != column.strip() for column in df.columns)
    string_space_rows = {}
    for column in df.select_dtypes(include=['object', 'string']).columns:
        values = df[column].dropna().astype(str)
        count = int(values.ne(values.str.strip()).sum())
        if count > 0:
            string_space_rows[column] = count
    print(f'{name}: 컬럼명 공백={column_space_count}, 문자열 값 공백={string_space_rows}')

In [ ]:
expected_numeric_columns = {
    '우체국 피해사례': ['연령대', '최초 접수년', '최초 접수월', '피해액', '자료기준일'],
}

for name, columns in expected_numeric_columns.items():
    df = raw_dataframes[name]
    print(f'\n[{name}]')
    for column in columns:
        converted = pd.to_numeric(df[column], errors='coerce')
        new_missing = int((converted.isna() & df[column].notna()).sum())
        print(f'{column}: 현재 dtype={df[column].dtype}, 숫자 변환 실패={new_missing}건')

In [ ]:
print('[피해금액 단위 및 범위 확인]')
print('우체국 피해액 단위: 원')
display(df_postal_raw['피해액'].describe().to_frame())
print('피해액 0원 이하:', int(df_postal_raw['피해액'].le(0).sum()), '건')

## 0-9. 최소 전처리

원본 DataFrame을 보호하기 위해 복사본에서만 작업합니다. 컬럼명과 문자열의 앞뒤 공백을 안전하게 제거하고, 숫자형이어야 하는 컬럼만 명시적으로 변환합니다. 의미가 다른 범주를 임의로 합치거나 행을 삭제하지 않습니다.

In [ ]:
df_postal_clean = df_postal_raw.copy()

prepared_dataframes = {
    '우체국 피해사례 정리본': df_postal_clean,
}

for df in prepared_dataframes.values():
    df.columns = df.columns.str.strip()
    for column in df.select_dtypes(include=['object', 'string']).columns:
        df[column] = df[column].str.strip()

In [ ]:
postal_numeric = ['연령대', '최초 접수년', '최초 접수월', '피해액', '자료기준일']

for column in postal_numeric:
    df_postal_clean[column] = pd.to_numeric(df_postal_clean[column], errors='coerce')

print('숫자형 변환 후 새 결측 확인')
print('우체국 피해사례 정리본', int(df_postal_clean.isna().sum().sum()))

## 0-10. 우체국 파생변수 판단

`연령대`, `최초 접수년`, `최초 접수월`이 이미 존재하므로 재생성하지 않습니다. 고액피해 컬럼은 없지만 최신 `우체국_가설_검정대상.md`에는 확정된 금액 기준이 없습니다. 따라서 3천만 원 등의 기준을 임의 적용하지 않고, 기준 확정 전까지 고액피해 변수를 생성하지 않습니다.

In [ ]:
required_postal_columns = [
    '연령대', '피해자 성별', '최초 접수년', '최초 접수월', '피해액',
    '사기유형', '사칭기관', '전화_보이스피싱', '중복후보',
]
missing_required_columns = [
    column for column in required_postal_columns
    if column not in df_postal_clean.columns
]
high_loss_columns = [column for column in df_postal_clean.columns if '고액' in column]

print('필수 컬럼 누락:', missing_required_columns)
print('현재 고액피해 관련 컬럼:', high_loss_columns)
print('고액피해 변수 생성: 보류(최신 문서에 확정 기준 없음)')

## 0-11. 우체국 분석대상 데이터 준비

서비스 범위에 맞게 `전화_보이스피싱=True`이면서 `사기유형!='투자사기'`인 사례만 선택하고, 이미 표시된 `중복후보=True`는 분석용 복사본에서 제외합니다. 원본 `df_postal_raw`과 정리본 `df_postal_clean`은 그대로 유지됩니다. 현재 투자사기 43건은 모두 `전화_보이스피싱=False`이지만, 서비스 범위를 코드에 명확히 남기기 위해 두 조건을 모두 사용합니다.

> `중복후보`는 사건 식별자가 없는 상태에서 확정 중복을 뜻하지 않을 수 있습니다. 따라서 1단계 전에 제외 기준이 적절한지 사람이 다시 확인해야 합니다.

In [ ]:
phone_mask = df_postal_clean['전화_보이스피싱'].eq(True)
investment_mask = df_postal_clean['사기유형'].eq('투자사기')
duplicate_candidate_mask = df_postal_clean['중복후보'].eq(True)

df_postal_analysis = df_postal_clean[
    phone_mask & ~investment_mask & ~duplicate_candidate_mask
].copy()

postal_filter_counts = {
    '원본 표본 수': len(df_postal_raw),
    '전화 보이스피싱': int(phone_mask.sum()),
    '전화 외 사례': int((~phone_mask).sum()),
    '투자사기': int(investment_mask.sum()),
    '전체 중복후보': int(duplicate_candidate_mask.sum()),
    '전화 사례 중 중복후보 제외': int((phone_mask & duplicate_candidate_mask).sum()),
    '최종 분석 표본': len(df_postal_analysis),
}

for label, count in postal_filter_counts.items():
    print(f'{label}: {count}건')

In [ ]:
analysis_category_columns = ['연령대', '피해자 성별', '사기유형', '사칭기관']
for column in analysis_category_columns:
    print(f'\n[최종 분석 표본 - {column}]')
    display(df_postal_analysis[column].value_counts(dropna=False).to_frame('건수'))

## 0-12. 전처리 결과 최종 확인

전처리 전후 shape, 결측, 완전중복, dtype과 생성 변수를 비교합니다. 큰 피해액은 핵심 분석 대상이므로 IQR 밖의 값도 삭제하지 않습니다.

In [ ]:
final_dataframes = {
    '우체국 분석용': df_postal_analysis,
}

comparison_rows = []
raw_for_comparison = {
    '우체국 분석용': df_postal_raw,
}

for name, final_df in final_dataframes.items():
    raw_df = raw_for_comparison[name]
    comparison_rows.append({
        '데이터': name,
        '전처리 전 shape': str(raw_df.shape),
        '전처리 후 shape': str(final_df.shape),
        '최종 결측': int(final_df.isna().sum().sum()),
        '최종 완전중복 추가 행': int(final_df.duplicated().sum()),
        '원본에서 삭제한 행': 0,
    })

display(pd.DataFrame(comparison_rows))

In [ ]:
print('[최종 dtype]')
for name, df in final_dataframes.items():
    print(f'\n{name}')
    print(df.dtypes.to_string())

print('\n[최종 생성 변수]')
print('우체국: 생성 없음(고액피해 기준 미확정)')
print('범주 통일: 실제 표기 차이가 확인되지 않아 적용하지 않음')
print('이상치 삭제: 0건')
print('원본 CSV 저장/덮어쓰기: 수행하지 않음')

## 0-13. 0단계 요약

아래 셀은 우체국 분석 표본의 실제 숫자를 계산해 요약합니다. 중복후보 제외 기준과 고액피해 기준은 사람이 확인해야 합니다.

In [ ]:
print('=' * 60)
print('0단계 우체국 데이터 준비 결과')
print('=' * 60)
print('\n[우체국 피해사례]')
for label, count in postal_filter_counts.items():
    print(f'{label}: {count}건')
print('최종 결측:', int(df_postal_analysis.isna().sum().sum()), '개')
print('최종 완전중복 추가 행:', int(df_postal_analysis.duplicated().sum()), '건')
print('고액피해 변수: 생성 보류(최신 문서에 확정 기준 없음)')
print('처리한 내용: 전화 기반 사례 선택, 투자사기와 중복후보를 분석용 복사본에서 제외')
print('원본에서 삭제한 행: 0건')

print('\n[사람이 확인할 사항]')
print('1. 중복후보 37건 중 전화 사례 35건을 제외하는 기준의 적절성')
print('2. U3/U7/U8에 사용할 고액피해 금액 기준')
print('3. 표본 수가 매우 작은 사기유형·사칭기관 범주의 향후 처리 방법')

print('=' * 60)
print('0단계 완료')
print('다음 단계: 1. 기본 EDA')
print('=' * 60)

## **0단계 우체국 데이터 준비 완료**

우체국 데이터의 로드, 구조 확인, 결측·중복·범주·dtype 확인과 전화 기반 분석 표본 준비를 완료했습니다.

# 1. 우체국 기본 EDA

0단계에서 준비한 `df_postal_analysis`를 사용해 피해자 특성, 피해금액, 사기유형과 사칭기관의 기본 현황을 확인합니다.

## 1-1. 시각화 환경과 한글 폰트 설정

Google Colab에 나눔고딕을 한 번만 설치한 뒤 현재 런타임의 Matplotlib에 직접 등록합니다. 이 방식은 일반적으로 런타임 재시작이 필요 없습니다. 설치 셀 실행 후에도 한글이 깨지면 셀을 한 번 더 실행하세요.

In [ ]:
!apt-get update -qq
!apt-get install -qq fonts-nanum

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('Matplotlib 한글 폰트:', plt.rcParams['font.family'])

## 1-2. 우체국 분석 표본과 피해자 특성

0단계에서 전화 기반 보이스피싱과 중복후보 제외 조건으로 만든 `df_postal_analysis`만 사용합니다. 각 범주의 건수와 전체 분석 표본에서 차지하는 비율을 함께 확인합니다.

In [ ]:
def make_frequency_table(series, sort_index=False):
    """범주별 건수와 비율을 같은 표에서 확인하기 위한 작은 함수입니다."""
    counts = series.value_counts(dropna=False, sort=not sort_index)
    if sort_index:
        counts = counts.sort_index()
    table = counts.rename('건수').to_frame()
    table['비율(%)'] = (table['건수'] / table['건수'].sum() * 100).round(2)
    return table

postal_age_table = make_frequency_table(df_postal_analysis['연령대'], sort_index=True)
postal_gender_table = make_frequency_table(df_postal_analysis['피해자 성별'])
postal_fraud_type_table = make_frequency_table(df_postal_analysis['사기유형'])
postal_impersonation_table = make_frequency_table(df_postal_analysis['사칭기관'])

print('최종 분석 표본 수:', len(df_postal_analysis), '건')
print('\n[연령대]')
display(postal_age_table)
print('[성별]')
display(postal_gender_table)
print('[사기유형]')
display(postal_fraud_type_table)
print('[사칭기관]')
display(postal_impersonation_table)

In [ ]:
display(postal_age_table)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(postal_age_table.index.astype(str), postal_age_table['건수'])
ax.set_title('우체국 분석 표본의 연령대별 피해자 수')
ax.set_xlabel('연령대')
ax.set_ylabel('피해자수(명)')
plt.tight_layout()
plt.show()

In [ ]:
display(postal_gender_table)

fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(postal_gender_table.index.astype(str), postal_gender_table['건수'])
ax.set_title('우체국 분석 표본의 성별 피해자 수')
ax.set_xlabel('성별')
ax.set_ylabel('피해자수(명)')
plt.tight_layout()
plt.show()

## 1-3. 우체국 피해금액 분포

피해금액은 연속형 숫자 데이터이므로 범주별 막대그래프가 아니라 **금액 구간별 사례 수를 나타내는 히스토그램**으로 분포를 확인합니다. 표본 수, 평균, 중앙값, 최소·최대, 표준편차, Q1, Q3, IQR, IQR 이상치 상한과 이상치 후보 개수도 함께 확인합니다.

우체국 `피해액` 원본은 원 단위이며 시각화에서만 `피해액(원) / 10,000`으로 계산한 만원 단위를 사용합니다. 경찰청 피해금액의 억원 단위와 통합하거나 두 출처의 절대금액을 직접 비교하지 않습니다.

> 피해금액은 극단적으로 큰 값이 실제 고액 보이스피싱 피해일 수 있습니다. IQR 기준을 벗어난 값은 데이터 오류나 제거 대상으로 단정하지 않고, 이후 고액피해 분석에서 별도로 확인할 중요한 사례로 유지합니다.

In [ ]:
loss_describe = df_postal_analysis['피해액'].describe()
loss_q1 = loss_describe['25%']
loss_q3 = loss_describe['75%']
loss_iqr = loss_q3 - loss_q1
loss_lower_fence = loss_q1 - 1.5 * loss_iqr
loss_upper_fence = loss_q3 + 1.5 * loss_iqr
loss_skewness = df_postal_analysis['피해액'].skew()
loss_outlier_mask = (
    df_postal_analysis['피해액'].lt(loss_lower_fence)
    | df_postal_analysis['피해액'].gt(loss_upper_fence)
)
loss_outlier_count = int(loss_outlier_mask.sum())

postal_loss_summary = pd.DataFrame({
    '항목': [
        '표본 수', '평균', '중앙값', '최소값', '최대값', '표준편차',
        'Q1', 'Q3', 'IQR', 'IQR 이상치 상한', 'IQR 기준 이상치 개수', '왜도',
    ],
    '값': [
        loss_describe['count'], loss_describe['mean'], df_postal_analysis['피해액'].median(),
        loss_describe['min'], loss_describe['max'], loss_describe['std'],
        loss_q1, loss_q3, loss_iqr, loss_upper_fence, loss_outlier_count, loss_skewness,
    ],
    '단위': ['건', '원', '원', '원', '원', '원', '원', '원', '원', '원', '건', '없음'],
})

display(postal_loss_summary.style.format({'값': '{:,.2f}'}))
print(f"평균 피해액: {df_postal_analysis['피해액'].mean():,.0f}원")
print(f"중앙 피해액: {df_postal_analysis['피해액'].median():,.0f}원")
print(f"IQR 이상치 상한: {loss_upper_fence:,.0f}원")
print(f"IQR 기준 이상치 후보: {loss_outlier_count:,}건")

loss_outlier_amount_table = (
    df_postal_analysis.loc[loss_outlier_mask, '피해액']
    .value_counts()
    .sort_index()
    .rename_axis('피해액(원)')
    .to_frame('건수')
)
display(loss_outlier_amount_table)

In [ ]:
WON_PER_MANWON = 10_000
loss_amount_manwon = df_postal_analysis['피해액'].dropna() / WON_PER_MANWON

postal_loss_unit_check = pd.DataFrame({
    '피해액(원)': [10_000_000, 100_000_000],
    '피해금액(만원)': [10_000_000 / WON_PER_MANWON, 100_000_000 / WON_PER_MANWON],
})
display(postal_loss_unit_check)
assert postal_loss_unit_check['피해금액(만원)'].tolist() == [1_000, 10_000]

histogram_counts, histogram_edges = np.histogram(loss_amount_manwon, bins=15)
postal_loss_histogram_table = pd.DataFrame({
    '구간_시작(만원)': histogram_edges[:-1],
    '구간_끝(만원)': histogram_edges[1:],
    '건수': histogram_counts,
})
display(postal_loss_histogram_table)

# 연속형 금액을 구간화한 빈도표를 구간 폭 그대로 붙여 그리므로 일반 범주 막대그래프가 아니라 히스토그램입니다.
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    postal_loss_histogram_table['구간_시작(만원)'],
    postal_loss_histogram_table['건수'],
    width=postal_loss_histogram_table['구간_끝(만원)'] - postal_loss_histogram_table['구간_시작(만원)'],
    align='edge', edgecolor='black',
)
ax.set_title('우체국 분석 표본의 피해금액 히스토그램')
ax.set_xlabel('피해금액(만원)')
ax.set_ylabel('사례 수(건)')
plt.tight_layout()
plt.show()

### 로그 변환 보조 히스토그램

전체 히스토그램은 초고액 사례까지 포함해 원자료의 전체 범위를 보여주지만, 강한 우측 왜도 때문에 비교적 낮은 금액대가 왼쪽에 압축될 수 있습니다. 아래 그래프는 `log1p(피해금액(만원))`을 사용해 그 구간을 더 자세히 확인합니다.

> 로그 변환은 이상치나 고액 피해 사례를 삭제한 것이 아니라, 오른쪽으로 긴 피해금액 분포를 시각적으로 확인하기 쉽게 만든 보조 시각화입니다. 원래 금액의 해석은 위 전체 히스토그램과 기초통계표를 기준으로 합니다.

In [ ]:
log_loss_amount = np.log1p(loss_amount_manwon)
log_histogram_counts, log_histogram_edges = np.histogram(log_loss_amount, bins=15)
postal_loss_log_histogram_table = pd.DataFrame({
    '로그구간_시작': log_histogram_edges[:-1],
    '로그구간_끝': log_histogram_edges[1:],
    '건수': log_histogram_counts,
})
display(postal_loss_log_histogram_table)

# 로그 변환 값도 연속 구간별 빈도를 계산한 뒤 같은 방식으로 히스토그램을 그립니다.
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(
    postal_loss_log_histogram_table['로그구간_시작'],
    postal_loss_log_histogram_table['건수'],
    width=(
        postal_loss_log_histogram_table['로그구간_끝']
        - postal_loss_log_histogram_table['로그구간_시작']
    ),
    align='edge', edgecolor='black',
)
ax.set_title('우체국 피해금액 로그 변환 히스토그램(보조)')
ax.set_xlabel('log1p(피해금액(만원))')
ax.set_ylabel('사례 수(건)')
plt.tight_layout()
plt.show()

In [ ]:
postal_loss_boxplot_data = df_postal_analysis[['피해액']].dropna().copy()
postal_loss_boxplot_data['피해액_만원'] = postal_loss_boxplot_data['피해액'] / WON_PER_MANWON
display(postal_loss_boxplot_data['피해액_만원'].describe().to_frame())

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.boxplot(postal_loss_boxplot_data['피해액_만원'], vert=False)
ax.set_title('우체국 분석 표본의 피해금액 박스플롯')
ax.set_xlabel('피해금액(만원)')
ax.set_yticks([])
plt.tight_layout()
plt.show()

### 일반 피해금액 영역 확대 박스플롯

전체 박스플롯에서 상자가 왼쪽에 작게 보이고 오른쪽에 점들이 나타나는 것은 코드 오류가 아니라, Q3에서 1.5×IQR을 넘는 고액 사례가 존재하는 우측 왜도 분포의 정상적인 표현입니다. 점은 IQR 기준 이상치 **후보**이며 실제 피해사례를 삭제하지 않습니다. 아래 그래프는 같은 전체 데이터를 사용하고 X축 표시 범위만 IQR 상한까지 확대합니다.

In [ ]:
postal_loss_zoom_range = pd.DataFrame({
    '표시_시작(만원)': [0],
    '표시_끝_IQR상한(만원)': [loss_upper_fence / WON_PER_MANWON],
    '그래프에_사용한_전체표본수': [len(postal_loss_boxplot_data)],
})
display(postal_loss_zoom_range)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.boxplot(postal_loss_boxplot_data['피해액_만원'], vert=False)
ax.set_xlim(0, loss_upper_fence / WON_PER_MANWON)
ax.set_title('우체국 피해금액 박스플롯(IQR 상한까지 확대)')
ax.set_xlabel('피해금액(만원)')
ax.set_yticks([])
plt.tight_layout()
plt.show()

### 분포 해석 범위

현재 데이터에서는 평균 피해금액이 중앙값보다 크고, 최대값이 Q3와 IQR 상한보다 크게 나타납니다. 전체 히스토그램과 박스플롯에서 오른쪽 꼬리와 고액 점들이 확인된다면 피해금액 분포가 우측으로 치우쳐 있고 일부 고액 피해 사례가 평균에 영향을 주는 모습으로 해석할 수 있습니다.

이는 1단계 EDA의 분포 관찰이며 피해금액이 커진 원인, 집단 간 차이 또는 통계적 유의성을 뜻하지 않습니다. IQR 밖의 사례도 실제 고액 보이스피싱 피해일 수 있으므로 제거하지 않고 이후 고액피해 기준 확정과 심층분석 대상으로 유지합니다.

## 1-4. 우체국 사기유형과 사칭기관

사기유형과 사칭기관의 건수·비율을 확인합니다. 표본 수가 적은 범주도 현재는 자동 통합하지 않고 그대로 표시합니다.

In [ ]:
display(postal_fraud_type_table)

fraud_type_plot_table = postal_fraud_type_table.sort_values('건수')
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fraud_type_plot_table.index.astype(str), fraud_type_plot_table['건수'])
ax.set_title('우체국 분석 표본의 사기유형별 피해사례 수')
ax.set_xlabel('사례 수(건)')
ax.set_ylabel('사기유형')
plt.tight_layout()
plt.show()

In [ ]:
display(postal_impersonation_table)

impersonation_plot_table = postal_impersonation_table.sort_values('건수')
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(impersonation_plot_table.index.astype(str), impersonation_plot_table['건수'])
ax.set_title('우체국 분석 표본의 사칭기관별 피해사례 수')
ax.set_xlabel('사례 수(건)')
ax.set_ylabel('사칭기관')
plt.tight_layout()
plt.show()

In [ ]:
small_fraud_types = postal_fraud_type_table[postal_fraud_type_table['건수'] <= 5]
small_impersonation_types = postal_impersonation_table[postal_impersonation_table['건수'] <= 5]

print('[표본 5건 이하 사기유형 - 통합하지 않고 기록만 함]')
display(small_fraud_types)
print('[표본 5건 이하 사칭기관 - 통합하지 않고 기록만 함]')
display(small_impersonation_types)

## 1-5. 고액피해 변수 확인

0단계에서는 현재 데이터에 고액피해 컬럼이 없고 최신 가설 문서에도 확정된 금액 기준이 없어 파생변수 생성을 보류했습니다. 따라서 0단계 결정을 변경하거나 임의 기준을 적용하지 않으며, 이번 1단계에서도 고액피해 건수·비율 그래프를 만들지 않습니다. 기준이 문서로 확정된 뒤 별도 요청에서 진행해야 합니다.

In [ ]:
high_loss_candidates = [
    column for column in df_postal_analysis.columns
    if '고액피해' in column
]

if high_loss_candidates:
    high_loss_column = high_loss_candidates[0]
    postal_high_loss_table = make_frequency_table(df_postal_analysis[high_loss_column])
    display(postal_high_loss_table)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(postal_high_loss_table.index.astype(str), postal_high_loss_table['건수'])
    ax.set_title('고액피해 여부별 사례 수')
    ax.set_xlabel('고액피해 여부')
    ax.set_ylabel('사례 수(건)')
    plt.tight_layout()
    plt.show()
else:
    postal_high_loss_table = None
    print('고액피해 컬럼이 없어 집계와 그래프를 보류합니다.')

## 1-6. 우체국 접수 시기 기본 빈도

현재 분석 표본은 2025년 7~12월에 한정되어 있습니다. 연도·월별 건수만 확인하며 짧은 기간의 변화에 의미를 과도하게 부여하지 않습니다. 본격적인 시간 변화 분석은 이후 단계에서 진행합니다.

In [ ]:
postal_year_table = make_frequency_table(df_postal_analysis['최초 접수년'], sort_index=True)
postal_month_table = make_frequency_table(df_postal_analysis['최초 접수월'], sort_index=True)

print('[최초 접수년]')
display(postal_year_table)
print('[최초 접수월]')
display(postal_month_table)

# 우체국 기본 EDA 요약

아래 셀은 실행된 우체국 집계표에서 실제 값을 계산해 요약합니다. 수치는 하드코딩하지 않으며 관찰된 현황만 기술합니다.

In [ ]:
top_postal_age = postal_age_table['건수'].idxmax()
top_postal_gender = postal_gender_table['건수'].idxmax()
top_fraud_type = postal_fraud_type_table['건수'].idxmax()
top_impersonation = postal_impersonation_table['건수'].idxmax()

print('=' * 65)
print('우체국 기본 EDA 요약')
print('=' * 65)
print(f"분석 표본 수: {len(df_postal_analysis):,}건")
print(f"가장 많은 연령대: {top_postal_age}대 ({int(postal_age_table.loc[top_postal_age, '건수'])}건)")
print(f"가장 많은 성별: {top_postal_gender} ({int(postal_gender_table.loc[top_postal_gender, '건수'])}건)")
print(f"평균 피해금액: {df_postal_analysis['피해액'].mean():,.0f}원")
print(f"중앙 피해금액: {df_postal_analysis['피해액'].median():,.0f}원")
print(f"최대 피해금액: {df_postal_analysis['피해액'].max():,.0f}원")
print(f"가장 많은 사기유형: {top_fraud_type} ({int(postal_fraud_type_table.loc[top_fraud_type, '건수'])}건)")
print(f"가장 많은 사칭기관: {top_impersonation} ({int(postal_impersonation_table.loc[top_impersonation, '건수'])}건)")
if postal_high_loss_table is None:
    print('고액피해 건수 및 비율: 기준 미확정으로 집계 보류')
else:
    print('고액피해 건수 및 비율: 위 고액피해 집계표 확인')
print(f"표본 5건 이하 사기유형: {small_fraud_types.index.astype(str).tolist()}")
print(f"표본 5건 이하 사칭기관: {small_impersonation_types.index.astype(str).tolist()}")

print('\n[다음 단계에서 확인할 관계 후보 - 아직 결론이 아님]')
print('- 사기유형별 피해금액 차이 여부')
print('- 사칭기관별 피해금액 차이 여부')
print('- 연령대별 피해금액 차이 여부')
print('- 성별 피해금액 차이 여부')
print('- 고액피해 기준 확정 후 연령대·사기유형·사칭기관별 구성 차이 여부')

print('\n[사람이 확인할 사항]')
print('- 고액피해 금액 기준')
print('- 5건 이하 소수 범주의 이후 분석 처리 방법')
print('- 2025년 7~12월에 한정된 우체국 표본의 대표성')

## **우체국 기본 EDA 완료**

우체국 분석 표본의 빈도, 비율, 피해금액 기초통계와 기본 분포를 확인했습니다. 고액피해 기준은 아직 확정되지 않았습니다.

# 2. 전체 관계 스크리닝

이번 단계에서는 0단계에서 확정한 `df_postal_analysis`를 그대로 사용하여 우체국 개별 피해사례의 변수 관계를 넓게 탐색합니다. 각 검정의 **raw p-value와 효과크기**를 하나의 결과표에 모으되, 아직 다중검정 보정이나 최종 유의성 판정은 하지 않습니다.

경찰청 자료는 연도·지역·연령대별 집계자료이므로 개인 단위 관측치처럼 취급하지 않고, 이번 단계에서는 새로운 통계검정을 적용하지 않습니다.

## 2-1. 분석 기준과 공통 설정

1단계에서 피해금액의 강한 우측 왜도와 고액 사례가 확인되었으므로 정규성을 강하게 가정하지 않는 비모수 검정을 우선합니다.

- 3개 이상 그룹과 피해금액: Kruskal-Wallis, epsilon-squared
- 2개 그룹과 피해금액: Mann-Whitney U, rank-biserial correlation
- 범주형 변수끼리: 카이제곱 독립성 검정, Cramér's V

`ALPHA=0.05`는 raw p-value의 1차 후보 표시만 위한 기준입니다. 이 단계의 후보는 최종 유의 관계가 아닙니다.

In [ ]:
from scipy import __version__ as scipy_version
from scipy.stats import chi2_contingency, kruskal, mannwhitneyu

ALPHA = 0.05
COMMON_INDEPENDENCE_NOTE = (
    '사건 간 독립성 가정; 0단계 중복후보 제외 기준의 타당성은 별도 확인 필요'
)
screening_rows = []

required_screening_columns = ['연령대', '피해자 성별', '피해액', '사기유형', '사칭기관']
missing_screening_columns = [
    column for column in required_screening_columns
    if column not in df_postal_analysis.columns
]
if missing_screening_columns:
    raise KeyError(f'2단계 필수 컬럼 누락: {missing_screening_columns}')

print('SciPy 버전:', scipy_version)
print('유의수준(ALPHA):', ALPHA)
print('분석 DataFrame: df_postal_analysis')
print('분석 표본 수:', len(df_postal_analysis))
print('필수 컬럼 누락:', missing_screening_columns)
print('고액피해 관련 컬럼:', [c for c in df_postal_analysis.columns if '고액피해' in c])

### 효과크기 해석 기준

효과크기는 지표별 참고 기준을 따로 사용합니다. 경계값은 절대적인 결론이 아니라 스크리닝을 위한 실무적 참고값입니다.

- epsilon-squared: 0.01 미만 매우 작음, 0.06 미만 작음, 0.14 미만 중간, 그 이상 큼
- rank-biserial correlation의 절대값: 0.10 미만 매우 작음, 0.30 미만 작음, 0.50 미만 중간, 그 이상 큼
- Cramér's V: 교차표의 작은 차원 `min(행-1, 열-1)`에 따라 Cohen 기준을 조정

카이제곱 검정은 일반적인 Cochran 기준인 `기대빈도 1 미만 없음`과 `기대빈도 5 미만 셀 20% 이하`를 진단합니다. 조건이 불안정해도 범주를 자동 통합하지 않고 raw p-value를 참고값으로 남깁니다.

In [ ]:
def make_amount_group_summary(data, group_column):
    """그룹별 피해액 표본 수와 기초통계를 검정 전에 확인합니다."""
    return (
        data.groupby(group_column, dropna=False)['피해액']
        .agg(표본수='count', 평균_피해액='mean', 중앙값_피해액='median', 최소값='min', 최대값='max')
        .sort_index()
    )

def epsilon_squared_kruskal(statistic, group_count, sample_count):
    """Kruskal-Wallis H 통계량에 대응하는 epsilon-squared를 계산합니다."""
    if sample_count <= group_count:
        return np.nan
    return max(0.0, (statistic - group_count + 1) / (sample_count - group_count))

def interpret_epsilon_squared(value):
    if pd.isna(value): return '계산 불가'
    if value < 0.01: return '매우 작음'
    if value < 0.06: return '작음'
    if value < 0.14: return '중간'
    return '큼'

def interpret_rank_biserial(value):
    magnitude = abs(value)
    if magnitude < 0.10: return '매우 작음'
    if magnitude < 0.30: return '작음'
    if magnitude < 0.50: return '중간'
    return '큼'

def cramers_v(chi2_statistic, sample_count, row_count, column_count):
    """카이제곱 통계량으로 Cramér's V를 계산합니다."""
    dimension = min(row_count - 1, column_count - 1)
    if sample_count == 0 or dimension <= 0:
        return np.nan
    return np.sqrt(chi2_statistic / (sample_count * dimension))

def interpret_cramers_v(value, row_count, column_count):
    """교차표 차원을 반영한 Cohen 참고 기준으로 Cramér's V를 해석합니다."""
    dimension = min(row_count - 1, column_count - 1)
    if pd.isna(value) or dimension <= 0:
        return '계산 불가'
    small = 0.10 / np.sqrt(dimension)
    medium = 0.30 / np.sqrt(dimension)
    large = 0.50 / np.sqrt(dimension)
    if value < small: return '매우 작음'
    if value < medium: return '작음'
    if value < large: return '중간'
    return '큼'

def first_candidate_label(raw_p_value):
    """raw p-value만으로 3단계 검토 대상을 표시하며 최종 판정은 하지 않습니다."""
    return '3단계 보정 검토' if raw_p_value < ALPHA else '현재 근거 약함'

def small_group_note(group_summary):
    small_groups = group_summary[group_summary['표본수'] < 5].index.astype(str).tolist()
    if small_groups:
        return f"소표본 그룹(<5): {small_groups}"
    return '표본 5건 미만 그룹 없음'

def add_small_effect_warning(note, raw_p_value, effect_interpretation):
    if raw_p_value < ALPHA and effect_interpretation in ['매우 작음', '작음']:
        return note + '; raw p는 작지만 효과크기가 작아 해석 주의'
    return note

In [ ]:
def run_chi_square_screening(test_id, relation, first_variable, second_variable, analysis_type):
    """교차표→카이제곱→기대빈도 진단→Cramér's V 순서로 실행하고 결과행을 추가합니다."""
    observed_table = pd.crosstab(
        df_postal_analysis[first_variable],
        df_postal_analysis[second_variable],
        dropna=False,
    )
    print(f'[{test_id}] 관측빈도: {relation}')
    display(observed_table)

    chi2_stat, raw_p, dof, expected = chi2_contingency(observed_table)
    expected_table = pd.DataFrame(expected, index=observed_table.index, columns=observed_table.columns)
    print('[기대빈도]')
    display(expected_table.style.format('{:.2f}'))

    expected_under_5 = int((expected < 5).sum())
    total_cells = int(expected.size)
    under_5_ratio = expected_under_5 / total_cells
    minimum_expected = float(expected.min())
    assumption_ok = minimum_expected >= 1 and under_5_ratio <= 0.20

    diagnostic_table = pd.DataFrame({
        '기대빈도_5미만_셀': [expected_under_5],
        '전체_셀': [total_cells],
        '5미만_비율(%)': [under_5_ratio * 100],
        '최소_기대빈도': [minimum_expected],
        '자유도': [dof],
        '가정충족여부': ['충족' if assumption_ok else '주의'],
    })
    display(diagnostic_table.style.format({'5미만_비율(%)': '{:.2f}', '최소_기대빈도': '{:.3f}'}))

    effect = cramers_v(chi2_stat, int(observed_table.to_numpy().sum()), *observed_table.shape)
    effect_interpretation = interpret_cramers_v(effect, *observed_table.shape)
    row_small = observed_table.sum(axis=1)[observed_table.sum(axis=1) < 5].index.astype(str).tolist()
    column_small = observed_table.sum(axis=0)[observed_table.sum(axis=0) < 5].index.astype(str).tolist()
    if row_small or column_small:
        sparse_note = f'소표본 범주(행): {row_small}; 소표본 범주(열): {column_small}'
    else:
        sparse_note = '관측 합계 5건 미만 범주 없음'
    expected_note = (
        f'기대빈도 5 미만 {expected_under_5}/{total_cells}셀({under_5_ratio:.1%}), '
        f'최소 기대빈도 {minimum_expected:.3f}'
    )
    if not assumption_ok:
        expected_note += '; 카이제곱 raw p-value는 참고값이며 해석 주의'
    note = add_small_effect_warning(
        expected_note + '; ' + sparse_note + '; ' + COMMON_INDEPENDENCE_NOTE,
        raw_p,
        effect_interpretation,
    )

    screening_rows.append({
        '검정ID': test_id,
        '관계': relation,
        '분석구분': analysis_type,
        '독립변수': first_variable,
        '종속변수': second_variable,
        '변수유형': '범주형 ↔ 범주형',
        '검정방법': '카이제곱 독립성 검정',
        '검정선택이유': '두 범주형 변수의 독립성 탐색; 기대빈도 진단을 함께 확인',
        '통계량': float(chi2_stat),
        '통계량_종류': 'chi-square',
        'raw_p_value': float(raw_p),
        '효과크기': float(effect),
        '효과크기_종류': "Cramér's V",
        '효과크기_해석': effect_interpretation,
        '표본수': int(observed_table.to_numpy().sum()),
        '그룹수': f'{observed_table.shape[0]}×{observed_table.shape[1]}',
        '가정충족여부': '충족' if assumption_ok else '주의',
        '주의사항': note,
        '1차후보': first_candidate_label(raw_p),
    })
    display(pd.DataFrame([screening_rows[-1]]))
    return observed_table, expected_table, diagnostic_table

## 2-2. U1 연령대 ↔ 피해금액

**연구 질문:** 연령대에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 연령대  
**종속변수:** 피해액

**분석 목적:** 연령을 AI 위험도 판단의 후보 변수로 추가 분석할 가치가 있는지 탐색한다.

연령대는 3개 이상 그룹이고 피해금액은 우측 왜도와 고액 사례가 있으므로 Kruskal-Wallis 검정을 사용합니다. 유의하더라도 어느 그룹끼리 다른지는 현재 단계에서 확인하지 않습니다.

In [ ]:
age_amount_summary = make_amount_group_summary(df_postal_analysis, '연령대')
display(age_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
age_groups = [
    group['피해액'].dropna().to_numpy()
    for _, group in df_postal_analysis.groupby('연령대', sort=True)
]
age_h, age_raw_p = kruskal(*age_groups)
age_effect = epsilon_squared_kruskal(age_h, len(age_groups), sum(len(group) for group in age_groups))
age_effect_interpretation = interpret_epsilon_squared(age_effect)
age_note = (
    '피해액의 우측 왜도와 고액 사례를 반영한 비모수 검정; '
    + small_group_note(age_amount_summary)
    + '; 분포 모양이 다르면 중앙값 차이로만 해석하지 않음; 사후검정 미실시; '
    + COMMON_INDEPENDENCE_NOTE
)
age_note = add_small_effect_warning(age_note, age_raw_p, age_effect_interpretation)

screening_rows.append({
    '검정ID': 'S1', '관계': '연령대 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '연령대', '종속변수': '피해액', '변수유형': '범주형(3그룹 이상) ↔ 연속형',
    '검정방법': 'Kruskal-Wallis', '통계량': float(age_h), '통계량_종류': 'H',
    '검정선택이유': '3개 이상 그룹이며 피해액의 우측 왜도와 고액 사례가 확인됨',
    'raw_p_value': float(age_raw_p), '효과크기': float(age_effect),
    '효과크기_종류': 'epsilon-squared', '효과크기_해석': age_effect_interpretation,
    '표본수': sum(len(group) for group in age_groups), '그룹수': len(age_groups),
    '가정충족여부': '대체로 충족', '주의사항': age_note,
    '1차후보': first_candidate_label(age_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-3. U2 성별 ↔ 피해금액

**연구 질문:** 성별에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 피해자 성별  
**종속변수:** 피해액

**분석 목적:** 성별을 위험 판단 후보 변수로 추가 분석할 가치가 있는지 탐색한다.

성별은 2개 그룹이며 피해금액의 왜도와 고액 사례가 커서 Mann-Whitney U 양측 검정을 사용합니다.

In [ ]:
gender_amount_summary = make_amount_group_summary(df_postal_analysis, '피해자 성별')
display(gender_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
gender_labels = sorted(df_postal_analysis['피해자 성별'].dropna().unique().tolist())
if len(gender_labels) != 2:
    raise ValueError(f'Mann-Whitney U 검정에는 2개 그룹이 필요합니다: {gender_labels}')
gender_group_1 = df_postal_analysis.loc[
    df_postal_analysis['피해자 성별'].eq(gender_labels[0]), '피해액'
].dropna().to_numpy()
gender_group_2 = df_postal_analysis.loc[
    df_postal_analysis['피해자 성별'].eq(gender_labels[1]), '피해액'
].dropna().to_numpy()
gender_u, gender_raw_p = mannwhitneyu(
    gender_group_1, gender_group_2, alternative='two-sided', method='asymptotic'
)
gender_effect = 2 * gender_u / (len(gender_group_1) * len(gender_group_2)) - 1
gender_effect_interpretation = interpret_rank_biserial(gender_effect)
gender_note = (
    f'그룹 순서: {gender_labels[0]} vs {gender_labels[1]}; 양수 효과크기는 첫 그룹의 순위가 더 큰 방향; '
    '동점이 존재할 수 있어 asymptotic 방식 사용; 분포 모양이 다르면 중앙값 차이로만 해석하지 않음; '
    + COMMON_INDEPENDENCE_NOTE
)
gender_note = add_small_effect_warning(gender_note, gender_raw_p, gender_effect_interpretation)

screening_rows.append({
    '검정ID': 'S2', '관계': '성별 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '피해자 성별', '종속변수': '피해액', '변수유형': '범주형(2그룹) ↔ 연속형',
    '검정방법': 'Mann-Whitney U', '통계량': float(gender_u), '통계량_종류': 'U',
    '검정선택이유': '2개 그룹이며 피해액의 우측 왜도와 고액 사례가 확인됨',
    'raw_p_value': float(gender_raw_p), '효과크기': float(gender_effect),
    '효과크기_종류': 'rank-biserial correlation',
    '효과크기_해석': gender_effect_interpretation,
    '표본수': len(gender_group_1) + len(gender_group_2), '그룹수': 2,
    '가정충족여부': '대체로 충족', '주의사항': gender_note,
    '1차후보': first_candidate_label(gender_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-4. U4 사기유형 ↔ 피해금액

**연구 질문:** 사기유형에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 사기유형  
**종속변수:** 피해액

**분석 목적:** 사기유형을 AI 위험도 판단 후보 변수로 추가 분석할 가치가 있는지 탐색한다.

일부 유형의 표본이 매우 적으므로 범주를 삭제·통합하지 않고 Kruskal-Wallis 검정 결과에 소표본 주의를 기록합니다.

In [ ]:
fraud_amount_summary = make_amount_group_summary(df_postal_analysis, '사기유형')
display(fraud_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
fraud_groups = [
    group['피해액'].dropna().to_numpy()
    for _, group in df_postal_analysis.groupby('사기유형', sort=True)
]
fraud_h, fraud_raw_p = kruskal(*fraud_groups)
fraud_effect = epsilon_squared_kruskal(fraud_h, len(fraud_groups), sum(len(group) for group in fraud_groups))
fraud_effect_interpretation = interpret_epsilon_squared(fraud_effect)
fraud_note = (
    '피해액의 우측 왜도와 고액 사례를 반영한 비모수 검정; '
    + small_group_note(fraud_amount_summary)
    + '; 그룹별 표본 불균형이 커서 해석 주의; 사후검정 미실시; '
    + COMMON_INDEPENDENCE_NOTE
)
fraud_note = add_small_effect_warning(fraud_note, fraud_raw_p, fraud_effect_interpretation)

screening_rows.append({
    '검정ID': 'S3', '관계': '사기유형 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '사기유형', '종속변수': '피해액', '변수유형': '범주형(3그룹 이상) ↔ 연속형',
    '검정방법': 'Kruskal-Wallis', '통계량': float(fraud_h), '통계량_종류': 'H',
    '검정선택이유': '3개 이상 그룹이며 피해액의 우측 왜도·고액 사례와 소표본 그룹이 확인됨',
    'raw_p_value': float(fraud_raw_p), '효과크기': float(fraud_effect),
    '효과크기_종류': 'epsilon-squared', '효과크기_해석': fraud_effect_interpretation,
    '표본수': sum(len(group) for group in fraud_groups), '그룹수': len(fraud_groups),
    '가정충족여부': '소표본 주의', '주의사항': fraud_note,
    '1차후보': first_candidate_label(fraud_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-5. U5 사칭기관 ↔ 피해금액

**연구 질문:** 사칭기관에 따라 피해금액 분포에 차이가 있는가?

**독립변수:** 사칭기관  
**종속변수:** 피해액

**분석 목적:** 사칭기관을 AI 위험도 및 독립검증 활성화 조건의 후보 변수로 탐색한다.

사칭기관은 3개 이상 그룹이며 소표본 기관이 포함되어 있어 Kruskal-Wallis 검정과 epsilon-squared를 계산하고 해석 주의를 기록합니다.

In [ ]:
impersonation_amount_summary = make_amount_group_summary(df_postal_analysis, '사칭기관')
display(impersonation_amount_summary.style.format({
    '평균_피해액': '{:,.0f}', '중앙값_피해액': '{:,.0f}',
    '최소값': '{:,.0f}', '최대값': '{:,.0f}',
}))

In [ ]:
impersonation_groups = [
    group['피해액'].dropna().to_numpy()
    for _, group in df_postal_analysis.groupby('사칭기관', sort=True)
]
impersonation_h, impersonation_raw_p = kruskal(*impersonation_groups)
impersonation_effect = epsilon_squared_kruskal(
    impersonation_h, len(impersonation_groups), sum(len(group) for group in impersonation_groups)
)
impersonation_effect_interpretation = interpret_epsilon_squared(impersonation_effect)
impersonation_note = (
    '피해액의 우측 왜도와 고액 사례를 반영한 비모수 검정; '
    + small_group_note(impersonation_amount_summary)
    + '; 그룹별 표본 불균형이 커서 해석 주의; 사후검정 미실시; '
    + COMMON_INDEPENDENCE_NOTE
)
impersonation_note = add_small_effect_warning(
    impersonation_note, impersonation_raw_p, impersonation_effect_interpretation
)

screening_rows.append({
    '검정ID': 'S4', '관계': '사칭기관 ↔ 피해금액', '분석구분': '핵심',
    '독립변수': '사칭기관', '종속변수': '피해액', '변수유형': '범주형(3그룹 이상) ↔ 연속형',
    '검정방법': 'Kruskal-Wallis', '통계량': float(impersonation_h), '통계량_종류': 'H',
    '검정선택이유': '3개 이상 그룹이며 피해액의 우측 왜도·고액 사례와 소표본 그룹이 확인됨',
    'raw_p_value': float(impersonation_raw_p), '효과크기': float(impersonation_effect),
    '효과크기_종류': 'epsilon-squared',
    '효과크기_해석': impersonation_effect_interpretation,
    '표본수': sum(len(group) for group in impersonation_groups),
    '그룹수': len(impersonation_groups), '가정충족여부': '소표본 주의',
    '주의사항': impersonation_note, '1차후보': first_candidate_label(impersonation_raw_p),
})
display(pd.DataFrame([screening_rows[-1]]))

## 2-6. U6 사기유형 ↔ 사칭기관

**연구 질문:** 사기유형과 사칭기관 사이에 연관성이 있는가?

**변수:** 사기유형, 사칭기관

**분석 목적:** 반복되는 보이스피싱 시나리오 조합을 탐색한다.

두 변수 모두 범주형이므로 교차표를 먼저 확인한 뒤 카이제곱 독립성 검정과 Cramér's V를 계산합니다. 기대빈도 조건이 불안정하면 raw p-value는 참고값으로만 기록합니다.

In [ ]:
fraud_impersonation_observed, fraud_impersonation_expected, fraud_impersonation_diagnostic = (
    run_chi_square_screening(
        'S5', '사기유형 ↔ 사칭기관', '사기유형', '사칭기관', '핵심'
    )
)

## 2-7. 추가 피해자 ↔ 사기수법 스크리닝

다음 네 관계는 향후 피해자-사기수법 분석을 위한 탐색용 후보입니다. 이번 단계에서는 교차표, raw p-value, Cramér's V와 기대빈도 문제까지만 확인하며 취약집단 결론을 내리지 않습니다.

### S6. 연령대 ↔ 사기유형

In [ ]:
age_fraud_observed, age_fraud_expected, age_fraud_diagnostic = run_chi_square_screening(
    'S6', '연령대 ↔ 사기유형', '연령대', '사기유형', '추가 탐색'
)

### S7. 연령대 ↔ 사칭기관

In [ ]:
age_impersonation_observed, age_impersonation_expected, age_impersonation_diagnostic = (
    run_chi_square_screening(
        'S7', '연령대 ↔ 사칭기관', '연령대', '사칭기관', '추가 탐색'
    )
)

### S8. 성별 ↔ 사기유형

In [ ]:
gender_fraud_observed, gender_fraud_expected, gender_fraud_diagnostic = run_chi_square_screening(
    'S8', '성별 ↔ 사기유형', '피해자 성별', '사기유형', '추가 탐색'
)

### S9. 성별 ↔ 사칭기관

In [ ]:
gender_impersonation_observed, gender_impersonation_expected, gender_impersonation_diagnostic = (
    run_chi_square_screening(
        'S9', '성별 ↔ 사칭기관', '피해자 성별', '사칭기관', '추가 탐색'
    )
)

## 2-8. 보류 가설 확인

### U3. 연령대 ↔ 고액피해 여부
고액피해 기준이 확정되지 않아 보류합니다.

### U7. 사기유형 ↔ 고액피해 여부
고액피해 기준이 확정되지 않아 보류합니다.

### U8. 사칭기관 ↔ 고액피해 여부
고액피해 기준이 확정되지 않아 보류합니다.

임의의 금액 기준을 만들지 않으며 이후 고액피해 분석 단계에서 기준 확정 후 검토합니다.

In [ ]:
held_hypotheses = pd.DataFrame([
    {'가설ID': 'U3', '관계': '연령대 ↔ 고액피해 여부', '상태': '보류', '사유': '고액피해 기준 미확정'},
    {'가설ID': 'U7', '관계': '사기유형 ↔ 고액피해 여부', '상태': '보류', '사유': '고액피해 기준 미확정'},
    {'가설ID': 'U8', '관계': '사칭기관 ↔ 고액피해 여부', '상태': '보류', '사유': '고액피해 기준 미확정'},
])
display(held_hypotheses)

## 2-9. 전체 `screening_results`

실행한 9개 검정을 하나의 DataFrame으로 정리합니다. `raw_p_value`는 다음 단계에서 그대로 사용할 수 있도록 숫자형을 유지하고, 화면 표시용 p-value 컬럼만 별도로 만듭니다.

In [ ]:
screening_results = pd.DataFrame(screening_rows)

if screening_results['검정ID'].duplicated().any():
    duplicate_ids = screening_results.loc[
        screening_results['검정ID'].duplicated(keep=False), '검정ID'
    ].tolist()
    raise ValueError(f'중복 검정ID 발견: {duplicate_ids}')
if len(screening_results) != 9:
    raise ValueError(f'예상한 검정 수는 9개이지만 {len(screening_results)}개가 기록되었습니다.')

screening_results['raw_p_value'] = pd.to_numeric(screening_results['raw_p_value'])
screening_results['통계량'] = pd.to_numeric(screening_results['통계량'])
screening_results['효과크기'] = pd.to_numeric(screening_results['효과크기'])

def display_p_value(value):
    return f'{value:.3e}' if value < 0.001 else f'{value:.4f}'

screening_results_display = screening_results.copy()
screening_results_display['통계량'] = screening_results_display['통계량'].map(lambda value: f'{value:.4f}')
screening_results_display['raw_p_value_표시'] = screening_results_display['raw_p_value'].map(display_p_value)
screening_results_display['효과크기'] = screening_results_display['효과크기'].map(lambda value: f'{value:.4f}')

display_columns = [
    '검정ID', '관계', '분석구분', '검정방법', '검정선택이유', '통계량', 'raw_p_value_표시',
    '효과크기', '효과크기_종류', '효과크기_해석', '표본수',
    '가정충족여부', '주의사항', '1차후보',
]
display(screening_results_display[display_columns])
print('\n[원본 결과 dtype - raw_p_value는 숫자형 유지]')
print(screening_results.dtypes)

## 2-10. 2단계 결과 요약

아래 셀은 실행 결과에서 총 검정 수, raw p-value 후보, 효과크기, 가정 및 소표본 주의를 자동 집계합니다. 현재 결과는 다중검정 보정 전 스크리닝 결과이므로 최종 유의 관계로 확정하지 않습니다.

In [ ]:
raw_candidate_results = screening_results[screening_results['raw_p_value'] < ALPHA]
weak_raw_results = screening_results[screening_results['raw_p_value'] >= ALPHA]
medium_or_larger_results = screening_results[
    screening_results['효과크기_해석'].isin(['중간', '큼'])
]
chi_square_caution_results = screening_results[
    screening_results['검정방법'].eq('카이제곱 독립성 검정')
    & screening_results['가정충족여부'].eq('주의')
]
small_sample_caution_results = screening_results[
    screening_results['주의사항'].str.contains('소표본', na=False)
]
small_effect_raw_candidates = raw_candidate_results[
    raw_candidate_results['효과크기_해석'].isin(['매우 작음', '작음'])
]

screening_summary = pd.DataFrame({
    '항목': [
        '총 실행 검정 수', 'raw p < 0.05 관계 수', '효과크기 중간 이상 관계 수',
        '카이제곱 가정 주의 관계 수', '표본 부족 주의 관계 수',
        'raw p는 작지만 효과크기가 작은 관계 수',
    ],
    '값': [
        len(screening_results), len(raw_candidate_results), len(medium_or_larger_results),
        len(chi_square_caution_results), len(small_sample_caution_results),
        len(small_effect_raw_candidates),
    ],
})
display(screening_summary)

print('[3단계 보정 검토 후보 - 아직 최종 유의 관계가 아님]')
print(raw_candidate_results['관계'].tolist())
print('\n[현재 raw p-value 근거가 약한 관계]')
print(weak_raw_results['관계'].tolist())
print('\n[효과크기 중간 이상 관계]')
print(medium_or_larger_results[['관계', '효과크기', '효과크기_종류', '효과크기_해석']].to_dict('records'))
print('\n[카이제곱 가정 주의 관계]')
print(chi_square_caution_results['관계'].tolist())
print('\n[소표본 주의 관계]')
print(small_sample_caution_results['관계'].tolist())
print('\n[raw p는 작지만 효과크기가 작은 관계]')
print(small_effect_raw_candidates['관계'].tolist())
print('\n[보류]')
for _, row in held_hypotheses.iterrows():
    print(f"- {row['가설ID']} {row['관계']}: {row['사유']}")

print('\n주의: 모든 p-value는 다중검정 보정 전 raw p-value입니다.')
print('3단계 보정 전에는 최종 유의 관계로 확정하지 않습니다.')

## **2단계 전체 관계 스크리닝 완료**

현재 Notebook에서는 우체국 개별 피해사례의 핵심 5개 관계와 추가 탐색 4개 관계에 대해 raw p-value, 효과크기와 가정 진단을 수집했습니다.

- 고액피해 관련 U3·U7·U8은 기준 미확정으로 보류했습니다.
- 경찰청 집계자료와 시간 변수에는 새로운 검정을 적용하지 않았습니다.
- 범주를 자동 삭제·통합하지 않았고 고액 피해 사례도 제거하지 않았습니다.
- 사후검정과 다중검정 보정은 수행하지 않았습니다.

Colab 실행 결과와 기대빈도·소표본 주의를 사람이 확인한 뒤 다음 요청에서 3단계 다중검정 보정을 진행합니다.